## Plan
explaining what we'll do in AG and ACO

## Fonctions utilitaires communes

---



In [31]:
import random
import requests
import time
import math
import numpy as np
from collections import defaultdict

In [32]:
#fonction recuperation du graphe du website DMACS
def lire_graphe_triangulaire(url_fichier,is_url):
    """
    Lit un fichier DIMACS et convertit le graphe en une matrice triangulaire inférieure.

    :param url_fichier: URL du fichier .col contenant le graphe
    :return: Matrice triangulaire inférieure sous forme de liste de listes
    """
    lignes = []
    if(is_url):
      response = requests.get(url_fichier)
      text = response.text
      lignes = text.split('\n')
    else:
      with open(url_fichier, 'r') as f:
        lignes = f.readlines()

    nb_sommets = 0
    aretes = []

    for ligne in lignes:
        ligne = ligne.strip()
        if ligne.startswith('p'):
            _, _, nb_sommets, _ = ligne.split()
            nb_sommets = int(nb_sommets)
        elif ligne.startswith('e'):
            _, sommet1, sommet2 = ligne.split()
            sommet1, sommet2 = int(sommet1), int(sommet2)
            aretes.append((sommet1, sommet2))

    # Création de la matrice triangulaire inférieure
    matrice_triangulaire = [[0] * (i + 1) for i in range(nb_sommets)]
    for sommet1, sommet2 in aretes:
        sommet1 -= 1  # Ajustement des indices (DIMACS commence à 1)
        sommet2 -= 1
        if sommet1 > sommet2:
            matrice_triangulaire[sommet1][sommet2] = 1
        else:
            matrice_triangulaire[sommet2][sommet1] = 1

    return matrice_triangulaire, nb_sommets

In [33]:
def neighbors_of(node, matrice):
  neighbors = set()
  for j in range(node):
    if matrice[node][j] == 1:
      neighbors.add(j)
  for j in range(node + 1, len(matrice)):
    if matrice[j][node] == 1:
      neighbors.add(j)
  return neighbors

In [34]:
#solution aléatoire (non basé sur une heuristique ou quoi que ce soit)
def initialize_solution(num_vertices, chromatic_number):
    """
    Generates a random vertex coloring.

    :param num_vertices: Number of vertices in the graph
    :param chromatic_number: Graph's chromatic number
    :return: Random initial coloring
    """
    return [random.randint(1, chromatic_number) for _ in range(num_vertices)]

In [35]:
#solution aléatoire (non basé sur une heuristique ou quoi que ce soit)
def random_solution(num_vertices):
    """
    Generates a random vertex coloring.

    :param num_vertices: Number of vertices in the graph
    :return: Random initial coloring
    """
    return [random.randint(1, num_vertices) for _ in range(num_vertices)]

In [36]:
#combien de sommets adjacents ont la même couleur
def calculate_conflicts(adj_matrix, num_vertices, coloring):
    """
    Calculates the number of coloring conflicts (edges with same color at both ends).

    :param adj_matrix: Graph adjacency matrix
    :param num_vertices: Number of vertices in the graph
    :param coloring: Current vertex coloring
    :return: Total number of conflicts
    """
    conflicts = 0
    for i in range(num_vertices):
        for j in range(i):
            if adj_matrix[i][j] == 1 and coloring[i] == coloring[j]:
                conflicts += 1
    return conflicts


In [37]:
def generate_neighbor(coloring, chromatic_number):
    """
    Generates a neighboring solution by randomly changing one vertex's color.

    :param coloring: Current coloring
    :param chromatic_number: Number of available colors
    :return: New neighboring coloring
    """
    neighbor = coloring.copy()
    vertex = random.randint(0, len(coloring) - 1)
    current_color = neighbor[vertex]

    # Choose a new different color
    new_color = current_color
    while new_color == current_color:
        new_color = random.randint(1, chromatic_number)

    neighbor[vertex] = new_color
    return neighbor

In [38]:
def fitness(adj_matrix, num_vertices, coloring):
    """
    Calculates fitness based on the number of non-conflicting vertices.
    Higher fitness is better.

    :param adj_matrix: Graph's adjacency matrix.
    :param num_vertices: Total number of vertices in the graph.
    :param coloring: List of assigned colors to vertices.
    :return: Fitness score (higher is better).
    """
    return num_vertices - calculate_conflicts(adj_matrix, num_vertices, coloring)


In [39]:
#dsatur algorithm four our constructing solutions in ACO
def get_neighbors(node, matrice_triangulaire,n):
        """
        Retourne l'ensemble des voisins du nœud 'node' en utilisant la matrice triangulaire.
        Pour un nœud i, les voisins sont ceux :
          - Dans la même ligne pour les indices j < i (si matrice[i][j] == 1)
          - Dans les lignes suivantes pour lesquels le nœud i apparaît (si matrice[k][i] == 1 pour k > i)
        """
        neighbors = set()
        # Pour les indices j < node, vérifier la ligne 'node'
        for j in range(node):
            if matrice_triangulaire[node][j] == 1:
                neighbors.add(j)
        # Pour les indices > node, le nœud 'node' se trouve en colonne dans la ligne k
        for k in range(node + 1, n):
            if matrice_triangulaire[k][node] == 1:
                neighbors.add(k)
        return neighbors

def dsatur(matrice_triangulaire):
  start = time.time()
  n = len(matrice_triangulaire)
  voisins = {i: get_neighbors(i, matrice_triangulaire,n) for i in range (n)}
  coloring = [None]*n # affectation initial
  ensemble_saturation = [set() for _ in range(n)] #initialisation des ensembles de saturation de chaque noeud
  degre = [len(voisins[i]) for i in range(n)] # Use a list, not a set
  # tantque on n'a pas fini de colorier otus les noeuds
  while any(color is None for color in coloring): # Loop until all nodes are colored
    non_colories = [i for i in range(n) if coloring[i] is None]
    if not non_colories:
          break
    chosen = max(non_colories, key=lambda x: (len(ensemble_saturation[x]), degre[x]))
    used_colors = {coloring[v] for v in voisins[chosen] if coloring[v] is not None}
    color = 0
    while color in used_colors:
      color += 1
    coloring[chosen] = color
    for v in voisins[chosen]:
        if coloring[v] is None:
            ensemble_saturation[v].add(color)
  chromatic_number = max(coloring) + 1 if coloring else 0
  temps = time.time() - start
  return chromatic_number, coloring, temps

# AG

## I. Representing a solution instance in a chromosomic manner

In [ ]:
class Chromosome:
    def __init__(self, coloring, adj_matrix, num_vertices):
        self.coloring = coloring
        self.fitness = fitness(adj_matrix, num_vertices, coloring)

    def __hash__(self):
        return hash(tuple(self.coloring))

    def __eq__(self, other):
        return self.coloring == other.coloring

    def __repr__(self):
        return f"Chromosome({self.coloring})"

In [ ]:
def generate_population(pop_size, adj_matrix, num_vertices):
    """
    Generates an initial population of chromosomes.

    :param pop_size : Population size (number of chromosomes).
    :param num_vertices : Number of graph vertices.

    :return list: List of randomly generated chromosomes.
    """
    population = []
    for _ in range(pop_size):
        coloring = random_solution(num_vertices)
        population.append(Chromosome(coloring, adj_matrix, num_vertices))
    return population

In [ ]:
matrix = [[0],
[1,0],
[0,1,0],
[1,0,1,0]]

solutions = generate_population(12, matrix,4)
for solution in solutions:
  print(solution.coloring,solution.fitness)

[4, 1, 1, 1] 2
[2, 2, 1, 2] 2
[4, 2, 4, 3] 4
[1, 2, 4, 3] 4
[3, 2, 2, 3] 2
[1, 1, 4, 1] 2
[3, 3, 3, 1] 2
[4, 1, 4, 1] 4
[3, 3, 2, 1] 3
[1, 2, 3, 1] 3
[2, 1, 4, 3] 4
[4, 3, 2, 3] 4


## II. Selecting subset of solutions

In [ ]:
def roulette_selection(population, num_chromo):
    """
    Selects a chromosome from the population using roulette wheel selection.
    The probability of selecting a chromosome is proportional to its fitness.

    :param population: List of chromosome objects.
    :param num_chromo: Number of chromosomes to select.
    :return: A selected chromosome.
    """
    selected = []
    remaining_pop = population.copy()

    for _ in range(num_chromo):
        total_fitness = sum(chromo.fitness for chromo in remaining_pop)
        if total_fitness <= 0:  # Handle case where all fitnesses are 0 or negative
            selected.extend(random.sample(remaining_pop, num_chromo - len(selected)))
            break

        probs = [chromo.fitness / total_fitness for chromo in remaining_pop]
        chosen_chromo = random.choices(range(len(remaining_pop)), weights=probs, k=1)[0]
        selected.append(remaining_pop.pop(chosen_chromo))

    return selected

In [ ]:
def elitism_selection(population, num_elites):
    """
    Selects the top N fittest chromosomes (elitism).

    :param population: List of Chromosome instances
    :param num_elites: Number of elites to retain
    :return: List of elite Chromosomes
    """
    sorted_population = sorted(population, key=lambda c: c.fitness, reverse=True)
    return sorted_population[:num_elites]

In [ ]:
def random_selection(population, num_chromo):
    """
    Randomly selects one chromosome from the population.

    :param population: List of Chromosome instances
    :param num_chromo: Number of chromosomes to select
    :return: Random Chromosome
    """
    selected = []
    remaining_pop = population.copy()
    for _ in range(num_chromo):
        chosen = random.choice(remaining_pop)
        selected.append(chosen)
        remaining_pop.remove(chosen)

    return selected

In [ ]:
matrix = [[0],
[1,0],
[0,1,0],
[1,0,1,0]]

solutions = generate_population(12, matrix,4)
print("======== Population ========")
print("|    Coloring    | Fitness |")
for solution in solutions:
  print("| ", solution.coloring, " |   ", solution.fitness, "   |")
print("============================\n")

print("========= Roulette =========")
print("|    Coloring    | Fitness |")
roul_sol = roulette_selection(solutions, 5)
for solution in roul_sol:
  print("| ", solution.coloring, " |   ", solution.fitness, "   |")
print("============================\n")

print("========= Elitism  =========")
print("|    Coloring    | Fitness |")
elitism_sol = elitism_selection(solutions, 5)
for solution in elitism_sol:
  print("| ", solution.coloring, " |   ", solution.fitness, "   |")
print("============================\n")

print("========= Random  =========")
print("|    Coloring    | Fitness |")
random_sol = random_selection(solutions, 5)
for solution in random_sol:
  print("| ", solution.coloring, " |   ", solution.fitness, "   |")
print("============================\n")

======== Population ========
|    Coloring    | Fitness |
|  [3, 2, 3, 1]  |    4    |
|  [2, 2, 2, 4]  |    2    |
|  [4, 3, 2, 3]  |    4    |
|  [1, 2, 1, 3]  |    4    |
|  [4, 3, 1, 2]  |    4    |
|  [3, 2, 4, 4]  |    3    |
|  [4, 2, 3, 2]  |    4    |
|  [2, 3, 4, 4]  |    3    |
|  [3, 2, 2, 4]  |    3    |
|  [1, 1, 1, 2]  |    2    |
|  [2, 4, 1, 4]  |    4    |
|  [4, 4, 3, 1]  |    3    |

========= Roulette =========
|    Coloring    | Fitness |
|  [2, 3, 4, 4]  |    3    |
|  [2, 2, 2, 4]  |    2    |
|  [2, 4, 1, 4]  |    4    |
|  [3, 2, 2, 4]  |    3    |
|  [4, 2, 3, 2]  |    4    |

========= Elitism  =========
|    Coloring    | Fitness |
|  [3, 2, 3, 1]  |    4    |
|  [4, 3, 2, 3]  |    4    |
|  [1, 2, 1, 3]  |    4    |
|  [4, 3, 1, 2]  |    4    |
|  [4, 2, 3, 2]  |    4    |

========= Random  =========
|    Coloring    | Fitness |
|  [3, 2, 4, 4]  |    3    |
|  [2, 2, 2, 4]  |    2    |
|  [4, 2, 3, 2]  |    4    |
|  [1, 1, 1, 2]  |    2    |
|  [1, 2, 1,

## III. Croisement

In [ ]:
def one_point_crossover(parent1: Chromosome, parent2: Chromosome, adj_matrix: list[list[int]], num_vertices: int) -> tuple[Chromosome, Chromosome]:
    """
    Effectue un croisement à un point entre deux parents.

    Args:
        parent1 (Chromosome): Premier parent
        parent2 (Chromosome): Deuxième parent
        adj_matrix (list[list[int]]): Matrice d'adjacence du graphe
        num_vertices (int): Nombre de sommets

    Returns:
        tuple[Chromosome, Chromosome]: Deux enfants issus du croisement
    """
    # Récupération des colorations des parents
    p1 = parent1.coloring.copy()
    p2 = parent2.coloring.copy()

    # Sélection aléatoire d'un point de coupure (entre 1 et n-1)
    crossover_point = random.randint(1, num_vertices - 1)

    # Création des enfants
    child1 = p1[:crossover_point] + p2[crossover_point:]
    child2 = p2[:crossover_point] + p1[crossover_point:]

    return Chromosome(child1, adj_matrix, num_vertices), Chromosome(child2, adj_matrix, num_vertices)

In [ ]:
def two_point_crossover(parent1: Chromosome, parent2: Chromosome, adj_matrix: list[list[int]], num_vertices: int) -> tuple[Chromosome, Chromosome]:
    """
    Effectue un croisement à deux points entre deux parents.

    Args:
        parent1 (Chromosome): Premier parent
        parent2 (Chromosome): Deuxième parent
        adj_matrix (list[list[int]]): Matrice d'adjacence du graphe
        num_vertices (int): Nombre de sommets

    Returns:
        tuple[Chromosome, Chromosome]: Deux enfants issus du croisement
    """
    p1 = parent1.coloring.copy()
    p2 = parent2.coloring.copy()

    # Génération de deux points distincts triés
    points = sorted(random.sample(range(num_vertices + 1), 2))
    point1, point2 = points

    # Création des enfants
    child1 = p1[:point1] + p2[point1:point2] + p1[point2:]
    child2 = p2[:point1] + p1[point1:point2] + p2[point2:]

    return Chromosome(child1, adj_matrix, num_vertices), Chromosome(child2, adj_matrix, num_vertices)

In [ ]:
def uniform_crossover(parent1: Chromosome, parent2: Chromosome, adj_matrix: list[list[int]], num_vertices: int, mix_prob: float = 0.5) -> tuple[Chromosome, Chromosome]:
    """
    Effectue un croisement uniforme entre deux parents.

    Args:
        parent1 (Chromosome): Premier parent
        parent2 (Chromosome): Deuxième parent
        adj_matrix (list[list[int]]): Matrice d'adjacence du graphe
        num_vertices (int): Nombre de sommets
        mix_prob (float, optional): Probabilité de prendre le gène du parent1. Defaults to 0.5.

    Returns:
        tuple[Chromosome, Chromosome]: Deux enfants issus du croisement
    """
    p1 = parent1.coloring.copy()
    p2 = parent2.coloring.copy()

    child1 = []
    child2 = []

    # Mélange aléatoire pour chaque gène
    for i in range(num_vertices):
        if random.random() < mix_prob:
            child1.append(p1[i])
            child2.append(p2[i])
        else:
            child1.append(p2[i])
            child2.append(p1[i])

    return Chromosome(child1, adj_matrix, num_vertices), Chromosome(child2, adj_matrix, num_vertices)

In [ ]:
def crossover(parent1: Chromosome, parent2: Chromosome, adj_matrix: list[list[int]], num_vertices: int,
             crossover_type: str = 'one_point', **kwargs) -> tuple[Chromosome, Chromosome]:
    """
    Applique l'opérateur de croisement spécifié entre deux parents.

    Args:
        parent1 (Chromosome): Premier parent
        parent2 (Chromosome): Deuxième parent
        adj_matrix (list[list[int]]): Matrice d'adjacence
        num_vertices (int): Nombre de sommets
        crossover_type (str): Type de croisement ('one_point', 'two_point', 'uniform')
        **kwargs: Paramètres additionnels pour le croisement

    Returns:
        tuple[Chromosome, Chromosome]: Les deux enfants produits
    """
    if crossover_type == 'one_point':
        return one_point_crossover(parent1, parent2, adj_matrix, num_vertices)
    elif crossover_type == 'two_point':
        return two_point_crossover(parent1, parent2, adj_matrix, num_vertices)
    elif crossover_type == 'uniform':
        mix_prob = kwargs.get('mix_prob', 0.5)
        return uniform_crossover(parent1, parent2, adj_matrix, num_vertices, mix_prob)
    else:
        raise ValueError(f"Type de croisement inconnu: {crossover_type}")

In [ ]:
# TESTS
def test_crossover():
    """
    Test complet des opérateurs de croisement avec visualisation des résultats
    """
    # Configuration reproductible
    random.seed(42)

    # Création d'un graphe test simple
    matrix_test = [
        [0],
        [1, 0],
        [1, 1, 0],
        [0, 1, 0, 0],
        [0, 0, 1, 1, 0]
    ]
    num_vertices = 5

    # Création de parents tests
    parent1 = Chromosome([1, 2, 1, 2, 3], adj_matrix, num_vertices)
    parent2 = Chromosome([2, 1, 3, 1, 2], adj_matrix, num_vertices)

    print("===== Test des opérateurs de croisement =====")
    print(f"Parent 1: {parent1.coloring} (Fitness: {parent1.fitness})")
    print(f"Parent 2: {parent2.coloring} (Fitness: {parent2.fitness})\n")

    # Test croisement à un point
    c1, c2 = one_point_crossover(parent1, parent2, matrix_test, num_vertices)
    print("[One-Point] Enfant 1:", c1.coloring, "(Fitness:", c1.fitness, ")")
    print("[One-Point] Enfant 2:", c2.coloring, "(Fitness:", c2.fitness, ")\n")

    # Test croisement à deux points
    c3, c4 = two_point_crossover(parent1, parent2, matrix_test, num_vertices)
    print("[Two-Point] Enfant 1:", c3.coloring, "(Fitness:", c3.fitness, ")")
    print("[Two-Point] Enfant 2:", c4.coloring, "(Fitness:", c4.fitness, ")\n")

    # Test croisement uniforme
    c5, c6 = uniform_crossover(parent1, parent2, matrix_test, num_vertices)
    print("[Uniform] Enfant 1:", c5.coloring, "(Fitness:", c5.fitness, ")")
    print("[Uniform] Enfant 2:", c6.coloring, "(Fitness:", c6.fitness, ")")

# Exécution du test
test_crossover()

===== Test des opérateurs de croisement =====
Parent 1: [1, 2, 1, 2, 3] (Fitness: 3)
Parent 2: [2, 1, 3, 1, 2] (Fitness: 4)

[One-Point] Enfant 1: [1, 1, 3, 1, 2] (Fitness: 3 )
[One-Point] Enfant 2: [2, 2, 1, 2, 3] (Fitness: 3 )

[Two-Point] Enfant 1: [2, 1, 1, 2, 3] (Fitness: 4 )
[Two-Point] Enfant 2: [1, 2, 3, 1, 2] (Fitness: 5 )

[Uniform] Enfant 1: [1, 2, 1, 1, 2] (Fitness: 4 )
[Uniform] Enfant 2: [2, 1, 3, 2, 3] (Fitness: 4 )


## IV. Mutation

In [ ]:
def mutate(chromosome, Pm, matrice, debug=False):

    """
     Input:
      :chromosome to mutate;
      :Pm; probability of mutating :chromosome
      :matrice graph
     Output: :chromosome mutated with a probability of :Pm
     Prioritizes conflicting nodes
    """

    if debug:
        print("----> Input: mutate")
        print(chromosome.coloring)

    if random.random() > Pm:
        if debug:
            print("----> Output: mutate")
            print("<No change>")
        return chromosome  # Retourner le chromosome inchangé

    coloring = chromosome.coloring.copy()
    conflicting_nodes = []
    num_vertices = len(matrice)

    # Find all conflicting nodes
    for i in range(num_vertices):
        for j in range(i):
            if matrice[i][j] == 1 and coloring[i] == coloring[j]:
                conflicting_nodes.extend([i, j])

    if not conflicting_nodes:
        # Si pas de conflits, mutation aléatoire
        node = random.randint(0, len(coloring)-1)
        current_color = coloring[node]
        all_colors = list(set(coloring))
        if len(all_colors) > 1:
            new_color = random.choice([c for c in all_colors if c != current_color])
            coloring[node] = new_color
    else:
        # Prioritiser la mutation des nœuds en conflit
        node = random.choice(conflicting_nodes)
        adjacent_colors = set()
        for j in range(node):
            if matrice[node][j] == 1:
                adjacent_colors.add(coloring[j])
        for j in range(node + 1, num_vertices):
            if matrice[j][node] == 1:
                adjacent_colors.add(coloring[j])

        # Choisir une couleur non utilisée par les voisins
        all_colors = list(set(coloring))
        available_colors = [c for c in all_colors if c not in adjacent_colors]
        if available_colors:
            coloring[node] = random.choice(available_colors)
        else:
            # Si toutes les couleurs sont utilisées par les voisins, en choisir une au hasard
            coloring[node] = random.choice(all_colors)

    if debug:
        print("<---- Output: mutate")
        print(coloring)

    # Créer et retourner un nouveau chromosome
    return Chromosome(coloring, matrice, num_vertices)

In [ ]:
import random

# Imaginons un tout petit graphe à 5 sommets
# Sommet 0 est connecté à 1 et 2
# Sommet 1 est connecté à 2 et 3
# Sommet 2 est connecté à 3
# etc.
matrice_test = [
    [0],
    [1, 0],
    [1, 1, 0],
    [0, 1, 1, 0],
    [0, 0, 0, 1, 0]
]
nb_sommets = 5

# Une solution (coloration) aléatoire pour commencer
coloring = [0, 0, 1, 2, 2]  # volontairement des conflits entre 0-1
print(f"Coloration initiale : {coloring}")

# On crée un chromosome
chromosome_test = Chromosome(coloring, matrice_test, nb_sommets)


# On fixe la seed pour reproductibilité
random.seed(42)

# Appel de la mutation avec Pm = 1 pour forcer la mutation
chromosome_mutated = mutates(chromosome_test, Pm=1.0, matrice=matrice_test, debug=True)

print(f"Coloration après mutation : {chromosome_mutated.coloring}")


Coloration initiale : [0, 0, 1, 2, 2]
----> Input: mutate
[0, 0, 1, 2, 2]
<---- Output: mutate
[0, 2, 1, 2, 2]
Coloration après mutation : [0, 2, 1, 2, 2]


In [ ]:
# On définit un dummy Chromosome si tu n'as pas encore fait
class Chromosome:
    def __init__(self, coloring):
        self.coloring = coloring

    def __repr__(self):
        return f"Chromosome({self.coloring})"

# On crée une petite population d'exemples
population = [
    Chromosome([1, 2, 3, 1]),
    Chromosome([2, 1, 1, 3]),
    Chromosome([3, 3, 2, 2]),
]

# On définit un masque pour le masque_crossover
masque = [0, 1, 0, 1]

# On définit une matrice d'adjacence pour le mutate
matrice = [
    [0, 1, 0, 0],  # Node 0 is connected to Node 1
    [1, 0, 1, 0],  # Node 1 is connected to Node 0 and 2
    [0, 1, 0, 1],  # Node 2 is connected to Node 1 and 3
    [0, 0, 1, 0],  # Node 3 is connected to Node 2
]

# Tester mutate
print("\n=== Test mutate ===")
mutated = mutate(population[0], Pm=1.0, matrice=matrice, debug=True)  # Pm=1.0 pour forcer la mutation
print("Mutated chromosome:", mutated)



=== Test mutate ===
----> Input: mutate
[1, 2, 3, 1]
<---- Output: mutate
[1, 1, 3, 1]


TypeError: Chromosome.__init__() takes 2 positional arguments but 4 were given

## V. Update

In [ ]:
#Zineb's code
def remplacement_strict(population, new_generation, size, matrice):
    # Combine la population actuelle et la nouvelle génération
    combined = population + new_generation
    # Trie la population combinée par fitness (meilleur en premier)
    combined.sort(key=lambda c: c.fitness, reverse=True)
    # Retourne les meilleurs 'size' individus
    return combined[:size]


In [ ]:
def remplacement_elitisme(population, new_generation, size, matrice, elitism_count):

    # Filtrer les None de la population
    population = [p for p in population if p is not None]
    new_generation = [p for p in new_generation if p is not None]

    # S'il n'y a pas assez d'individus, compléter avec des chromosomes aléatoires
    num_vertices = len(matrice)
    while len(population) + len(new_generation) < size:
        coloring = [random.randint(1, len(set(population[0].coloring)))
                   for _ in range(num_vertices)]
        new_generation.append(Chromosome(coloring, matrice, num_vertices))

    # Sélectionner les meilleurs chromosomes
    elite = elitism_selection(population, min(elitism_count, len(population)))

    # Compléter avec les meilleurs chromosomes de la population combinée
    combined = population + new_generation
    remaining = elitism_selection(combined, size - len(elite))

    return elite + remaining

In [ ]:
def remplacement_random(population, new_generation, size):
    # Sélectionne un nombre aléatoire de chromosomes parmi la population actuelle
    selected = random_selection(population, size - len(new_generation))
    # Combine les nouveaux enfants avec les chromosomes sélectionnés au hasard
    new_population = new_generation + selected
    return new_population


In [ ]:
def remplacement_les_moins_forts(population, new_generation, size):
    # Trie la population en fonction de la fitness (du plus fort au plus faible)
    sorted_population = sorted(population, key=lambda c: c.fitness, reverse=True)
    # Conserve les meilleurs individus
    best_individuals = sorted_population[:len(population) - len(new_generation)]
    # Remplace les plus faibles par les enfants
    new_population = best_individuals + new_generation
    return new_population


In [ ]:
def remplacement(current_population: list[Chromosome], next_gen: list[Chromosome]) -> list[Chromosome]:
    '''
    Update method: replace all parents by next_gen
    '''
    return next_gen


## Main AG

In [ ]:
def genetic_algorithm(population, matrice, num_vertices, max_generations, crossover_rate,
                     mutation_rate, elitism_count, target_colors, population_size):
    """
    Exécute l'algorithme génétique pour la coloration de graphe.
    """
    # Vérification de la population initiale
    population = [p for p in population if p is not None]
    if not population:
        # Créer une population si celle-ci est vide
        population = generate_population(population_size, matrice, num_vertices, target_colors)

    best_solution = max(population, key=lambda c: c.fitness)
    best_fitness = best_solution.fitness
    no_improvement_count = 0
    convergence_gen = 0

    for generation in range(max_generations):
        # Sélection des parents
        parents = roulette_selection(population, max(population_size // 2, 2))
        # parents = random_selection(population, max(population_size // 2, 2))


        # Création de la nouvelle génération
        new_generation = []

        # Élitisme - conserver les meilleurs individus
        elite = elitism_selection(population, min(elitism_count, len(population)))
        new_generation.extend(elite)

        # Croisement et mutation
        while len(new_generation) < population_size:
            # Sélection de deux parents
            if len(parents) < 2:
                # Compléter parents si nécessaire
                while len(parents) < 2:
                    coloring = [random.randint(1, target_colors) for _ in range(num_vertices)]
                    parents.append(Chromosome(coloring, matrice, num_vertices))

            parent1, parent2 = random.sample(parents, 2)

            # Croisement avec une certaine probabilité
            if random.random() < crossover_rate:
                # child1, child2 = uniform_crossover(parent1, parent2, matrice, num_vertices)
                 child1, child2 = one_point_crossover(parent1, parent2, matrice, num_vertices)
                # child1, child2 = two_point_crossover(parent1, parent2, matrice, num_vertices)
            else:
                child1, child2 = parent1, parent2

            # Mutation
            child1 = mutate(child1, mutation_rate, matrice)
            child2 = mutate(child2, mutation_rate, matrice)

            # Ajout à la nouvelle génération
            new_generation.append(child1)
            if len(new_generation) < population_size:
                new_generation.append(child2)

        # Limitation de la taille de la population
        new_generation = new_generation[:population_size]

        # Remplacement de la population
        population = remplacement_elitisme(population, new_generation, population_size, matrice, elitism_count)
        #population = remplacement_strict(population, new_generation, population_size, matrice)
        #population = remplacement_random(population, new_generation, population_size, matrice)
        #population = remplacement_les_moins_forts(population, new_generation, population_size, matrice)

        # Mise à jour de la meilleure solution
        current_best = max(population, key=lambda c: c.fitness)
        if current_best.fitness > best_fitness:
            best_solution = current_best
            best_fitness = current_best.fitness
            no_improvement_count = 0
            convergence_gen = generation
        else:
            no_improvement_count += 1

        # Affichage de la progression
        if generation % 50 == 0:
            used_colors = len(set(best_solution.coloring))
            conflicts = calculate_conflicts(matrice, num_vertices, best_solution.coloring)
            print(f"Génération {generation}: Fitness = {best_fitness}, Couleurs = {used_colors}, Conflits = {conflicts}")

        # Vérification des critères d'arrêt
        best_colors = len(set(best_solution.coloring))
        best_conflicts = calculate_conflicts(matrice, num_vertices, best_solution.coloring)

        # Arrêt si solution optimale trouvée
        if best_conflicts == 0 :
            break
        else:
         # Arrêt si pas d'amélioration depuis 100 générations
         if no_improvement_count >= 100:
            print(f"Arrêt anticipé après {generation} générations sans amélioration")
            break

    return best_solution, convergence_gen, best_fitness


In [ ]:
import random
import requests
import time
import statistics

def main():
    # Dictionnaire des graphes à tester avec leur nombre chromatique connu
    graphs_best = {
        "dsjc125.9": 44,
        "dsjc125.1": 5,
        "dsjc250.1": 8,
        "r250.1": 8,
        "flat300_28_0": 28,
        "le450_25c": 25
    }

    # Paramètres de l'algorithme génétique
    population_size = 100
    max_generations = 1000
    crossover_rate = 0.8
    mutation_rate = 0.1
    elitism_count = 5

    results = {}

    for graph_name, best_color in graphs_best.items():
        print(f"\n===== Test sur le graphe {graph_name} =====")
        print(f"Nombre chromatique attendu: {best_color}")

        # Chargement du graphe
        try:
            start_time = time.time()
            matrice, number_nodes = lire_graphe_triangulaire(f"https://cedric.cnam.fr/~porumbed/graphs/{graph_name}.col", is_url=True)

            if number_nodes == 0:
                print(f"Échec du chargement du graphe {graph_name}")
                continue

            print(f"Graphe chargé: {number_nodes} sommets")

            # Affichage des 5 premières lignes de la matrice pour vérification
            print("Aperçu de la matrice:")
            for i in range(min(5, number_nodes)):
                print(matrice[i][:min(5, i+1)])

            # Initialisation de la population
            population = generate_population(population_size, matrice, number_nodes)

            # Vérification de la population initiale
            for i, chromo in enumerate(population):
                if chromo is None:
                    print(f"Warning: Chromosome {i} est None!")
                    # Remplacer le chromosome None par un valide
                    population[i] = Chromosome([random.randint(1, best_color) for _ in range(number_nodes)],
                                             matrice, number_nodes)

            # Exécution de l'algorithme génétique
            best_solution, convergence_gen, best_fitness = genetic_algorithm(
                population, matrice, number_nodes,
                max_generations, crossover_rate, mutation_rate,
                elitism_count, best_color,
                population_size
            )

            # Calcul du nombre de couleurs utilisées
            used_colors = len(set(best_solution.coloring))
            conflicts = calculate_conflicts(matrice, number_nodes, best_solution.coloring)

            # Temps d'exécution
            execution_time = time.time() - start_time

            # Stockage des résultats
            results[graph_name] = {
                "expected_colors": best_color,
                "used_colors": used_colors,
                "conflicts": conflicts,
                "fitness": best_solution.fitness,
                "convergence_gen": convergence_gen,
                "time": execution_time
            }

            # Affichage des résultats
            print(f"Résultats pour {graph_name}:")
            print(f"  - Couleurs attendues: {best_color}")
            print(f"  - Couleurs utilisées: {used_colors}")
            print(f"  - Conflits restants: {conflicts}")
            print(f"  - Fitness: {best_solution.fitness}")
            print(f"  - Convergence à la génération: {convergence_gen}")
            print(f"  - Temps d'exécution: {execution_time:.2f} secondes")

        except Exception as e:
            print(f"Erreur lors du traitement de {graph_name}: {e}")
            import traceback
            traceback.print_exc()

    # Résumé des résultats
    print("\n===== Résumé des résultats =====")
    for graph_name, result in results.items():
        success = "✓" if result["conflicts"] == 0 and result["used_colors"] <= result["expected_colors"] else "✗"
        print(f"{graph_name}: {success} {result['used_colors']}/{result['expected_colors']} couleurs, {result['conflicts']} conflits, {result['time']:.2f}s")

if __name__ == "__main__":
    main()


===== Test sur le graphe dsjc125.9 =====
Nombre chromatique attendu: 44
Graphe chargé: 125 sommets
Aperçu de la matrice:
[0]
[1, 0]
[1, 1, 0]
[1, 1, 1, 0]
[1, 1, 1, 1, 0]
Génération 0: Fitness = -74, Couleurs = 72, Conflits = 55
Génération 50: Fitness = -29, Couleurs = 70, Conflits = 14
Résultats pour dsjc125.9:
  - Couleurs attendues: 44
  - Couleurs utilisées: 65
  - Conflits restants: 0
  - Fitness: -5
  - Convergence à la génération: 89
  - Temps d'exécution: 7.01 secondes

===== Test sur le graphe dsjc125.1 =====
Nombre chromatique attendu: 5
Graphe chargé: 125 sommets
Aperçu de la matrice:
[0]
[0, 0]
[0, 0, 0]
[0, 0, 0, 0]
[1, 0, 0, 0, 0]
Génération 0: Fitness = -24, Couleurs = 71, Conflits = 7
Résultats pour dsjc125.1:
  - Couleurs attendues: 5
  - Couleurs utilisées: 65
  - Conflits restants: 0
  - Fitness: -5
  - Convergence à la génération: 14
  - Temps d'exécution: 1.11 secondes

===== Test sur le graphe dsjc250.1 =====
Nombre chromatique attendu: 8
Graphe chargé: 250 somme

# ACO

on commence par avoir dsatur qui va permettre de obtenir une solution initiale rapide et déterminer un nb de couleurs de départ réaliste pour ACO.

P(c) = [tau(sommet,c)^alpha] * [eta(c)^beta] / Σ [tau(sommet,j)^alpha] * [eta(j)^beta]

In [50]:
def valider_solution(matrice_triangulaire, solution):
    n = len(solution)
    for i in range(n):
        for j in range(i + 1):
            if matrice_triangulaire[i][j] == 1 and solution[i] == solution[j]:
                return False
    return True

In [42]:
def est_adjacent(matrice, i, j):
    if i == j:
        return 0
    elif i > j:
        return matrice[i][j]
    else:
        return matrice[j][i]

In [44]:
def calcul_degres(matrice_triangulaire, n):
    """
    Calcule le degré de chaque sommet pour une matrice triangulaire inférieure.
    """
    degres = [0] * n
    for i in range(n):
        for j in range(i + 1):
            if matrice_triangulaire[i][j] == 1:
                degres[i] += 1
                degres[j] += 1  # Car c'est symétrique
    return degres

## Optimisations notables dans notre implementation d'ACO

Les sommets de plus haut degré sont traités en premier (heuristique de saturation)

Dans l'heuristique on favorise les couleurs déjà utilisées pour minimiser le nombre total de couleurs

Paramètre q0 : Contrôle l'équilibre entre exploitation (utiliser ce qui fonctionne) et exploration (essayer de nouvelles options)

In [51]:
def algo_principal_aco_gcp(matrice_adjacence, nb_sommets, nb_fourmis=10, nb_iterations=100,
                           alpha=1.0, beta=2.0, rho=0.1, q0=0.9, nb_couleurs_max=None):
    """
    Implémentation de l'algorithme ACO pour GCP.

    Paramètres:
    - matrice_adjacence: matrice d'adjacence du graphe
    - nb_sommets: nombre de sommets dans le graphe
    - nb_fourmis: nombre de fourmis utilisées à chaque itération
    - nb_iterations: nombre max d'itérations
    - alpha: importance des phéromones
    - beta: importance de l'heuristique
    - rho: taux d'évaporation des phéromones
    - q0: paramètre d'exploitation vs exploration
    - nb_couleurs_max: nombre maximum de couleurs (si None, sera estimé)

    Retourne:
    - solution: liste des couleurs attribuées à chaque sommet
    - temps d'exécution en secondes
    """
    debut = time.time()

    # Estimation du nombre de couleurs si non fourni
    if nb_couleurs_max is None:
        degres = calcul_degres(matrice_adjacence, nb_sommets)
        degre_max = max(degres)
        nb_couleurs_max = min(degre_max + 1, nb_sommets)

    # Initialisation de la matrice de phéromones
    # Pour chaque sommet i et couleur j, tau[i][j] représente la désirabilité
    tau = np.ones((nb_sommets, nb_couleurs_max))

    # Meilleure solution trouvée
    meilleure_solution = None
    meilleur_nb_couleurs = float('inf')

    # Degrés des sommets (utilisés comme heuristique)
    degres = calcul_degres(matrice_adjacence, nb_sommets)

    # Ordre de visite des sommets (par degré décroissant)
    ordre_sommets = sorted(range(nb_sommets), key=lambda x: degres[x], reverse=True)

    for iteration in range(nb_iterations):
        solutions_fourmis = []

        for k in range(nb_fourmis):
            # Initialiser la solution pour cette fourmi
            solution = [-1] * nb_sommets
            couleurs_utilisees = set()

            # Colorer les sommets dans l'ordre défini
            for sommet in ordre_sommets:
                # Couleurs déjà utilisées par les voisins
                couleurs_voisins = set()
                for voisin in range(nb_sommets):
                    if est_adjacent(matrice_adjacence, sommet, voisin) == 1 and solution[voisin] != -1:
                        couleurs_voisins.add(solution[voisin])

                # Couleurs disponibles
                couleurs_disponibles = [c for c in range(nb_couleurs_max) if c not in couleurs_voisins]

                if not couleurs_disponibles:
                    # Aucune couleur disponible, solution non valide
                    solution = None
                    break

                # Calcul des probabilités de sélection
                proba = np.zeros(len(couleurs_disponibles))
                for i, couleur in enumerate(couleurs_disponibles):
                    # Heuristique: favoriser les couleurs déjà utilisées
                    eta = 2.0 if couleur in couleurs_utilisees else 1.0
                    proba[i] = (tau[sommet][couleur] ** alpha) * (eta ** beta)

                # Normaliser les probabilités
                if np.sum(proba) > 0:
                    proba = proba / np.sum(proba)
                else:
                    proba = np.ones(len(couleurs_disponibles)) / len(couleurs_disponibles)

                # Sélection de la couleur selon la règle de transition
                if random.random() < q0:  # Exploitation
                    couleur_choisie = couleurs_disponibles[np.argmax(proba)]
                else:  # Exploration
                    couleur_choisie = np.random.choice(couleurs_disponibles, p=proba)

                solution[sommet] = couleur_choisie
                couleurs_utilisees.add(couleur_choisie)

            if solution:
                solutions_fourmis.append(solution)

        # Mise à jour des phéromones
        tau = (1 - rho) * tau  # Évaporation

        for solution in solutions_fourmis:
            # Calculer le nombre de couleurs utilisées
            nb_couleurs = len(set(solution))

            # Mise à jour des phéromones
            delta_tau = 1.0 / nb_couleurs
            for sommet, couleur in enumerate(solution):
                tau[sommet][couleur] += delta_tau

            # Mettre à jour la meilleure solution
            if meilleure_solution is None or nb_couleurs < meilleur_nb_couleurs:
                meilleure_solution = solution.copy()
                meilleur_nb_couleurs = nb_couleurs

        # Terminer si on trouve une solution optimale
        if meilleur_nb_couleurs <= nb_couleurs_max:
            break

    fin = time.time()
    temps_execution = fin - debut

    return meilleure_solution, temps_execution

## Optimisations notables dans notre implementation d'ACO avec DSatur
une version améliorée qui intègre l'algorithme DSATUR (Degree of Saturation) avec ACO

On utilise une matrice de phéromones tau[sommet1][sommet2] entre paires de sommets non adjacents. Les phéromones sont nulles entre sommets adjacents (logique, car ils ne peuvent pas partager la même couleur). Cette approche favorise le regroupement de sommets non adjacents dans la même classe de couleur.

Le premier algorithme utilise un ordre statique basé sur le degré des sommets, l'ordre est déterminé une fois au début et ne change pas pendant l'exécution mais dans cet algorithme on implémente la stratégie dynamique DSATUR qui sélectionne le prochain sommet à colorier selon : le degré de saturation (nombre de couleurs différentes déjà attribuées aux voisins) et en cas d'égalité, le degré du sommet.

L'ordre est recalculé après chaque coloration de sommet, ce qui est généralement plus efficace, mais ajoute de la complexite en termes de temps d'execution

L'ordre est recalculé après chaque coloration de sommet, ce qui est généralement plus efficace.

### Implications des différences et pourquoi cet approche donne des resultats plus proches de l'optimal que la premiere
DSATUR est une des meilleures heuristiques pour la coloration de graphe

La représentation des phéromones favorise la formation de classes de couleur stables

L'ordre dynamique de traitement des sommets s'adapte mieux à la structure du graphe

In [58]:
def algo_principal_aco_gcp_dsat(matrice_adjacence, nb_sommets, nb_fourmis=10, nb_iterations=100,
                           alpha=1.0, beta=2.0, rho=0.1, q0=0.9, nb_couleurs_max=None):
    """
    Implémentation améliorée de l'algorithme ACO pour GCP avec DSATUR.

    Paramètres:
    - matrice_adjacence: matrice d'adjacence du graphe
    - nb_sommets: nombre de sommets dans le graphe
    - nb_fourmis: nombre de fourmis utilisées à chaque itération
    - nb_iterations: nombre max d'itérations
    - alpha: importance des phéromones
    - beta: importance de l'heuristique
    - rho: taux d'évaporation des phéromones
    - q0: paramètre d'exploitation vs exploration
    - nb_couleurs_max: nombre maximum de couleurs (si None, sera estimé)

    Retourne:
    - solution: liste des couleurs attribuées à chaque sommet
    - temps d'exécution en secondes
    """
    debut = time.time()

    # Estimation du nombre de couleurs si non fourni
    if nb_couleurs_max is None:
        degres = calcul_degres(matrice_adjacence, nb_sommets)
        degre_max = max(degres)
        nb_couleurs_max = min(degre_max + 1, nb_sommets)

    # Initialisation de la matrice de phéromones entre paires de sommets non adjacents
    tau = np.ones((nb_sommets, nb_sommets))
    for i in range(nb_sommets):
        for j in range(i+1, nb_sommets):
            if est_adjacent(matrice_adjacence, i, j) == 1:  # Si les sommets sont adjacents
                tau[i][j] = tau[j][i] = 0.0  # Pas de phéromone

    # Meilleure solution trouvée
    meilleure_solution = None
    meilleur_nb_couleurs = float('inf')

    # Degrés des sommets
    degres = calcul_degres(matrice_adjacence, nb_sommets)

    for iteration in range(nb_iterations):
        solutions_fourmis = []

        for k in range(nb_fourmis):
            solution = [-1] * nb_sommets
            couleurs_utilisees = set()

            # Tant qu'il reste des sommets non colorés
            while -1 in solution:
                # Calculer les degrés de saturation (DSAT) pour les sommets non colorés
                dsat = np.zeros(nb_sommets, dtype=int)
                for v in range(nb_sommets):
                    if solution[v] == -1:
                        couleurs_voisins = set()
                        for u in range(nb_sommets):
                            if  est_adjacent(matrice_adjacence, v, u) == 1 and solution[u] != -1:
                                couleurs_voisins.add(solution[u])
                        dsat[v] = len(couleurs_voisins)

                # Choisir le sommet avec le DSAT le plus élevé (et degré maximal en cas d'égalité)
                sommets_non_colores = [v for v in range(nb_sommets) if solution[v] == -1]
                sommet = max(sommets_non_colores, key=lambda x: (dsat[x], degres[x]))

                # Couleurs disponibles pour ce sommet
                couleurs_voisins = set()
                for u in range(nb_sommets):
                    if est_adjacent(matrice_adjacence, sommet, u) == 1 and solution[u] != -1:
                        couleurs_voisins.add(solution[u])
                couleurs_disponibles = [c for c in range(nb_couleurs_max) if c not in couleurs_voisins]

                if not couleurs_disponibles:
                    solution = None
                    break

                # Calcul des probabilités de sélection
                proba = np.zeros(len(couleurs_disponibles))
                for i, couleur in enumerate(couleurs_disponibles):
                    # Heuristique: favoriser les couleurs les moins utilisées (comme DSATUR)
                    eta = 1.0 / (1 + sum(1 for v in range(nb_sommets) if solution[v] == couleur))

                    # Phéromone: somme des phéromones avec les sommets non adjacents déjà colorés avec cette couleur
                    pheromone_sum = sum(
                        tau[sommet][u]
                        for u in range(nb_sommets)
                        if solution[u] == couleur and est_adjacent(matrice_adjacence, sommet, u) == 0
                    )

                    proba[i] = (pheromone_sum ** alpha) * (eta ** beta)

                # Normaliser les probabilités
                if np.sum(proba) > 0:
                    proba = proba / np.sum(proba)
                else:
                    proba = np.ones(len(couleurs_disponibles)) / len(couleurs_disponibles)

                # Sélection de la couleur
                if random.random() < q0:  # Exploitation
                    couleur_choisie = couleurs_disponibles[np.argmax(proba)]
                else:  # Exploration
                    couleur_choisie = np.random.choice(couleurs_disponibles, p=proba)

                solution[sommet] = couleur_choisie
                couleurs_utilisees.add(couleur_choisie)

            if solution:
                solutions_fourmis.append(solution)

        # Mise à jour des phéromones
        tau = (1 - rho) * tau  # Évaporation

        for solution in solutions_fourmis:
            nb_couleurs = len(set(solution))
            delta_tau = 1.0 / nb_couleurs

            # Mise à jour des phéromones pour les paires de sommets non adjacents partageant la même couleur
            for u in range(nb_sommets):
                for v in range(u+1, nb_sommets):
                    if est_adjacent(matrice_adjacence, u, v) == 0 and solution[u] == solution[v]:
                        tau[u][v] += delta_tau
                        tau[v][u] += delta_tau

            # Mettre à jour la meilleure solution
            if meilleure_solution is None or nb_couleurs < meilleur_nb_couleurs:
                meilleure_solution = solution.copy()
                meilleur_nb_couleurs = nb_couleurs

        # Terminer si on trouve une solution optimale
        if meilleur_nb_couleurs <= nb_couleurs_max:
            break

    fin = time.time()
    temps_execution = fin - debut

    return meilleure_solution, temps_execution

## Test

In [59]:
graphs_bench = {
    "dsjc125.9": 44,
    "dsjc125.1": 5,
    "dsjc250.1": 8,
    "r250.1": 8,
    "flat300_28_0": 28,
    "le450_25c": 25
}

for graph, borne_inf in graphs_bench.items():
    print(f"\n=== Résultats pour {graph} ===")
    matx, n = lire_graphe_triangulaire(f"https://cedric.cnam.fr/~porumbed/graphs/{graph}.col", is_url=True)

    # Appel de l'algorithme ACO
    solution, exec_time = algo_principal_aco_gcp_dsat(matx, n, nb_fourmis=20, nb_iterations=200, alpha=1.0, beta=2.0, rho=0.1, q0=0.9)

    if solution and valider_solution(matx, solution):
        nb_couleurs = len(set(solution))
        print(f"Meilleure solution: {nb_couleurs} couleurs")
        print(f"Borne inférieure: {borne_inf} couleurs")
        print(f"Écart: {nb_couleurs - borne_inf} couleurs")
        print(f"Temps d'exécution: {exec_time:.2f} secondes")
    else:
        print("Aucune solution valide trouvée.")


=== Résultats pour dsjc125.9 ===
Meilleure solution: 51 couleurs
Borne inférieure: 44 couleurs
Écart: 7 couleurs
Temps d'exécution: 5.72 secondes

=== Résultats pour dsjc125.1 ===
Meilleure solution: 6 couleurs
Borne inférieure: 5 couleurs
Écart: 1 couleurs
Temps d'exécution: 4.10 secondes

=== Résultats pour dsjc250.1 ===
Meilleure solution: 10 couleurs
Borne inférieure: 8 couleurs
Écart: 2 couleurs
Temps d'exécution: 26.53 secondes

=== Résultats pour r250.1 ===
Meilleure solution: 8 couleurs
Borne inférieure: 8 couleurs
Écart: 0 couleurs
Temps d'exécution: 22.86 secondes

=== Résultats pour flat300_28_0 ===
Meilleure solution: 39 couleurs
Borne inférieure: 28 couleurs
Écart: 11 couleurs
Temps d'exécution: 67.76 secondes

=== Résultats pour le450_25c ===
Meilleure solution: 28 couleurs
Borne inférieure: 25 couleurs
Écart: 3 couleurs
Temps d'exécution: 200.99 secondes


In [52]:
graphs_bench = {
    "dsjc125.9": 44,
    "dsjc125.1": 5,
    "dsjc250.1": 8,
    "r250.1": 8,
    "flat300_28_0": 28,
    "le450_25c": 25
}

for graph, borne_inf in graphs_bench.items():
    print(f"\n=== Résultats pour {graph} ===")
    matx, n = lire_graphe_triangulaire(f"https://cedric.cnam.fr/~porumbed/graphs/{graph}.col", is_url=True)

    # Appel de l'algorithme ACO
    solution, exec_time = algo_principal_aco_gcp(matx, n, nb_fourmis=20, nb_iterations=200,
                                                alpha=1.0, beta=2.0, rho=0.1, q0=0.9)

    if solution and valider_solution(matx, solution):
        nb_couleurs = len(set(solution))
        print(f"Meilleure solution: {nb_couleurs} couleurs")
        print(f"Borne inférieure: {borne_inf} couleurs")
        print(f"Écart: {nb_couleurs - borne_inf} couleurs")
        print(f"Temps d'exécution: {exec_time:.2f} secondes")
    else:
        print("Aucune solution valide trouvée.")


=== Résultats pour dsjc125.9 ===
Meilleure solution: 54 couleurs
Borne inférieure: 44 couleurs
Écart: 10 couleurs
Temps d'exécution: 0.36 secondes

=== Résultats pour dsjc125.1 ===
Meilleure solution: 8 couleurs
Borne inférieure: 5 couleurs
Écart: 3 couleurs
Temps d'exécution: 0.21 secondes

=== Résultats pour dsjc250.1 ===
Meilleure solution: 17 couleurs
Borne inférieure: 8 couleurs
Écart: 9 couleurs
Temps d'exécution: 0.73 secondes

=== Résultats pour r250.1 ===
Meilleure solution: 9 couleurs
Borne inférieure: 8 couleurs
Écart: 1 couleurs
Temps d'exécution: 0.41 secondes

=== Résultats pour flat300_28_0 ===
Meilleure solution: 48 couleurs
Borne inférieure: 28 couleurs
Écart: 20 couleurs
Temps d'exécution: 1.44 secondes

=== Résultats pour le450_25c ===
Meilleure solution: 44 couleurs
Borne inférieure: 25 couleurs
Écart: 19 couleurs
Temps d'exécution: 2.71 secondes


## Resources
1. https://www.researchgate.net/publication/4242538_Ant_Colony_System_for_Graph_Coloring_Problem
2. https://github.com/e0526847/graph-coloring/blob/master/ACO_graph_colouring.py


# Conclusion
AG et ACO remarks and stuff regarding the graph color problem